In [5]:
import pandas as pd
import numpy as np

df = pd.read_parquet('../data/processed/sapporo_density.parquet')

print("Rows:", len(df))
print("d range:", df["d"].min(), df["d"].max())
print("Unique days:", df["d"].nunique())


Rows: 6024808
d range: 0 74
Unique days: 75


In [6]:
# 假設 d=0 是 2023-01-01
df["date"] = pd.to_datetime("2023-01-01") + pd.to_timedelta(df["d"], unit="D")

df["weekday"] = df["date"].dt.weekday
df["is_weekend"] = (df["weekday"] >= 5).astype(int)


In [7]:
df = df.sort_values(["x","y","t","d"])

# 前一天
df["lag_1"] = df.groupby(["x","y","t"])["count"].shift(1)

# 前一週
df["lag_7"] = df.groupby(["x","y","t"])["count"].shift(7)

# 3日平均
df["rolling_3"] = (
    df.groupby(["x","y","t"])["count"]
      .transform(lambda x: x.shift(1).rolling(3).mean())
)

# 7日平均
df["rolling_7"] = (
    df.groupby(["x","y","t"])["count"]
      .transform(lambda x: x.shift(1).rolling(7).mean())
)

df = df.dropna()

print("After lag rows:", len(df))


After lag rows: 4881953


In [8]:
max_day = df["d"].max()

train_df = df[df["d"] <= max_day - 7]
test_df  = df[df["d"] > max_day - 7]

print("Train days:", train_df["d"].min(), "~", train_df["d"].max())
print("Test days:", test_df["d"].min(), "~", test_df["d"].max())


Train days: 7 ~ 67
Test days: 68 ~ 74


In [9]:
import lightgbm as lgb
from sklearn.metrics import r2_score

features = [
    "weekday", "t", "x", "y", "is_weekend",
    "lag_1", "lag_7", "rolling_3", "rolling_7"
]

train_sample = train_df.sample(n=800_000, random_state=42)

X_train = train_sample[features]
y_train = train_sample["count"]

X_test = test_df[features]
y_test = test_df["count"]

model = lgb.LGBMRegressor(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=4
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    eval_metric="l2",
    callbacks=[lgb.early_stopping(50)]
)

pred = model.predict(X_test)

print("Future Test R2:", r2_score(y_test, pred))


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.014645 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1063
[LightGBM] [Info] Number of data points in the train set: 800000, number of used features: 9
[LightGBM] [Info] Start training from score 3.248094
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1999]	valid_0's l2: 27.3524
Future Test R2: 0.9209368224207489


In [10]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(model, "../models/lgbm_sapporo.pkl")

print("✓ Model saved.")


✓ Model saved.
